# CS604 Colab launcher — Ana: LoRA r24 recipe ablation (pilot scale)

Which changes took LoRA from 0.717 to 0.785? Runs the arms in `configs/ablation_lora_r24.yaml`, each one change from `lora_r24_336_recipe.yaml`, on a 1,000-class pilot subset (40 fit images/class, 3 epochs) and scores them on a labelled 5/class holdout:

| Arm | Change |
|---|---|
| A | none (full recipe) |
| B | no norms / biases / pos_embed / cls_token |
| C | rank 16 instead of 24 |
| D | linear head from a linear probe (LP-FT) instead of the prototype cosine head |
| E | head lr 1e-4 instead of 3e-3 |
| F | B + C + D + E (≈ the 0.717 run's setup) |

Both starting heads are rebuilt from the pilot's fit images only, so no head has seen the holdout. Each arm runs in its own process. If the session drops, Run all again: finished arms are skipped.

About 1 hour on an A100: data download ~15 min, clean heads ~5 min, 6 arms × ~6 min. Results go to `CS604_RUNS/ablation_ablation_lora_r24/`.

Runtime: **A100 GPU**. Use **Runtime → Run all**.

In [ ]:
REPO_URL = "https://github.com/keaidewoq/cs604-project.git"
REPO_DIR = "/content/repo"
DATA_DIR = "/content/data"
# Must match a folder (or shortcut) in My Drive — not the browser URL.
DRIVE_OUT = "/content/drive/MyDrive/CS604_RUNS"
BRANCH = "ana-branch"
METHOD = "lora_r24_336_recipe"  # base config; only used to build prototypes_336.pt if it is missing
PROTO_PT = f"{DRIVE_OUT}/prototypes_336.pt"
SPEC = "configs/ablation_lora_r24.yaml"
PILOT_DATA_DIR = "/content/pilot_data"
FEATURE_DIR = "/content/features_pilot"  # local disk
# End the runtime after the last cell, or as soon as training fails.
AUTO_DISCONNECT = True


def end_runtime(reason):
    """Flush Drive writes, then release the GPU so it stops using compute units."""
    print(reason, flush=True)
    if not AUTO_DISCONNECT:
        return
    from google.colab import drive, runtime
    try:
        drive.flush_and_unmount()  # without this the last checkpoint can be lost
        print("Drive flushed and unmounted", flush=True)
    except Exception as error:
        print(f"Drive flush failed: {error}", flush=True)
    runtime.unassign()

## 1. Clone or pull the repo

In [ ]:
import os
from getpass import getpass

token = getpass("GitHub PAT (hidden; repo scope). Leave empty if the repo is public: ")
clone_url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    !git -C {REPO_DIR} fetch origin
    !git -C {REPO_DIR} checkout {BRANCH}
    if token:
        !git -C {REPO_DIR} pull {clone_url} {BRANCH}
    else:
        !git -C {REPO_DIR} pull origin {BRANCH}
else:
    !git clone -b {BRANCH} {clone_url} {REPO_DIR}

%cd {REPO_DIR}
!git branch --show-current
!git rev-parse --short HEAD

## 2. Install requirements

In [ ]:
%pip install -r requirements.txt

## 3. Mount Drive and check `CS604_RUNS`

If this cell errors, add a shortcut named `CS604_RUNS` to My Drive from the [shared folder](https://drive.google.com/drive/folders/107nc4GKekmaKe38GR_8VBn3Bh_xH-ZtR).

In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

out = Path(DRIVE_OUT)
if not out.is_dir():
    raise FileNotFoundError(
        f"{DRIVE_OUT} not found after mounting.\n"
        "Open https://drive.google.com/drive/folders/107nc4GKekmaKe38GR_8VBn3Bh_xH-ZtR "
        "→ Organize → Add shortcut to Drive → My Drive, and name the shortcut CS604_RUNS."
    )
out.mkdir(parents=True, exist_ok=True)
print("Writing runs to", out.resolve())
print("Existing items:", [p.name for p in out.iterdir()][:20])

## 4. Download data to `/content/data`

In [ ]:
!python scripts/download_data.py --dest {DATA_DIR}

## 4b. Build class prototypes at 336 (once)

Same as Van's prototype head, but built from 336 features (built with `configs/{METHOD}.yaml`, so at its `img_size`). Run **once**; about 20–25 min on an A100. Takes a while (embeds train images). Skip if the file already exists on Drive.

In [ ]:
from pathlib import Path

if Path(PROTO_PT).is_file():
    print("Already have", PROTO_PT)
else:
    !python scripts/build_prototypes.py --config configs/{METHOD}.yaml --data_root {DATA_DIR} --output {PROTO_PT}

## 5. Ablation

In [ ]:
import subprocess
import sys

proc = subprocess.run([sys.executable, "-u", "scripts/ablation.py", "--spec", SPEC,
                       "--data_root", DATA_DIR, "--output_dir", DRIVE_OUT,
                       "--pilot_data_dir", PILOT_DATA_DIR, "--feature_dir", FEATURE_DIR])
if proc.returncode != 0:
    end_runtime(f"ablation failed (exit {proc.returncode}); see the output above")
    raise SystemExit("ablation failed")

## 6. Results

`delta_vs_A_points`: change in pilot holdout top-1 (percentage points) against the full recipe. Negative = turning that piece off hurts, so that piece helped. Treat differences under ~0.3 points as ties (one seed).

In [ ]:
from pathlib import Path

import pandas as pd

table = pd.read_csv(Path(DRIVE_OUT) / "ablation_ablation_lora_r24" / "ablation.csv")
display(table[["arm", "changes", "final_val_acc", "delta_vs_A_points", "best_val_acc", "final_val_loss",
               "early_dip", "backbone_pct", "wall_min"]])

## 7. End the runtime

In [ ]:
end_runtime("all cells finished; ending the runtime")